# 영화 관객수 예측 – 부스팅 모델로 베이스라인 돌려보기 & 단계별 개선

**담당:** 부스팅 모델 (LightGBM · XGBoost · CatBoost)

## 이 노트북의 구성
| 파트 | 하는 일 | 왜? |
|---|---|---|
| **Part 1. 모델만 바꾸기** | 베이스라인 전처리는 그대로 두고, 랜덤포레스트 자리에 부스팅 3종을 넣어 비교 | 조원들과 **같은 조건**에서 "어떤 모델이 잘 맞는가"를 비교하기 위해 (맡은 역할) |
| **Part 2. 한 번에 하나씩 개선** | 전처리를 한 단계씩 바꿔 가며 점수 변화를 확인 | 무엇이 점수를 올렸는지 **원인을 정확히** 알기 위해 |
| **Part 3. 튜닝** | 모델별로 가장 좋은 단계에서 하이퍼파라미터 조정 | 마지막으로 점수를 더 끌어올리기 위해 |
| **Part 4. 결과 정리** | 로컬 점수와 리더보드 점수를 한 표로 정리 | 조원 공유 · 발표용 |

### 용어 정리
- **로컬 점수(CV RMSE):** 제출 전에 내 컴퓨터에서 매겨 보는 모의고사 점수. train을 5조각으로 나눠 4조각으로 공부하고 1조각으로 시험 보기를 5번 반복합니다.
- **리더보드 점수(LB):** 데이콘에 제출해서 받은 실제 점수.
- **RMSE:** 대략 "평균적으로 몇 명 정도 틀렸는지". **낮을수록 좋습니다.** 대박 영화를 크게 틀리면 점수가 확 나빠지는 특징이 있습니다.

### 실행 방법
1. Colab 왼쪽 파일 탭에 `movies_train.csv`, `movies_test.csv`, `submission.csv`를 올립니다.
2. `런타임 → 모두 실행`. 몇 분 걸립니다.
3. 만들어진 csv를 데이콘에 제출하고, 점수를 Part 4의 `KNOWN_LB`에 적으면 표가 자동으로 채워집니다.

---
## 0. 라이브러리 & 데이터 불러오기

**부스팅(Boosting)이란?** 작은 예측기(결정트리)를 **순서대로** 만들면서, 앞의 예측기가 틀린 부분을 다음 예측기가 고쳐 나가는 방법입니다(오답 노트 방식).
베이스라인의 **랜덤포레스트**는 결정트리를 **따로따로 동시에** 여러 개 만들어 평균을 냅니다(여러 명에게 답을 받아 평균 내는 방식).

| 모델 | 특징 |
|---|---|
| LightGBM | 마이크로소프트가 만듦. 빠르고 가벼움 |
| XGBoost | 가장 오래되고 유명한 부스팅. 대회에서 많이 쓰임 |
| CatBoost | 얀덱스가 만듦. 기본 설정만으로도 성능이 좋고 과적합에 강한 편 |

In [ ]:
!pip install -q catboost   # Colab에 catboost가 없을 때 설치

import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
warnings.filterwarnings('ignore')

SEED = 42   # 랜덤 시드 고정: 다시 돌려도 같은 결과

train_raw = pd.read_csv('movies_train.csv')    # 정답(관객수)이 있는 600편
test_raw  = pd.read_csv('movies_test.csv')     # 맞혀야 할 243편
submission = pd.read_csv('submission.csv')     # 제출 양식
print(train_raw.shape, test_raw.shape, submission.shape)

---
## 1. 전처리 함수

베이스라인 전처리를 함수 하나로 묶고, **옵션 두 개만 바꾸면** 개선 버전이 나오게 했습니다.

### 모든 버전 공통 (베이스라인과 동일)
- `title`(제목), `director`(감독 이름) 삭제
- `release_time`(개봉일)에서 **년, 월** 뽑기
- 장르 · 상영등급 같은 글자는 **원핫 인코딩** (예: 장르가 액션이면 `genre_액션=1`)

### 옵션 ① `bfnum` – 감독의 이전 영화 평균 관객수(`dir_prev_bfnum`)
600개 중 **330개가 비어 있습니다.** 비어 있는 건 대부분 **감독의 데뷔작**이라 이전 기록이 없기 때문입니다.
- `'drop'` : 베이스라인처럼 열을 삭제
- `'fill0'` : 빈칸을 "이전 관객 0명"으로 채워서 살림 (값 차이가 커서 log 변환도 함께)

### 옵션 ② `dist` – 배급사(`distributor`, 169종류)
- `'top5'` : 베이스라인처럼 큰 배급사 5개만 남기고 나머지는 '기타'
- `'rank'` : 배급사마다 **평소 관객수(중앙값) 순위**를 매겨 숫자 하나로 바꿈. 흥행을 잘 시키는 배급사일수록 큰 숫자. test에만 있는 배급사는 0

### 베이스라인에서 고친 점
베이스라인은 train과 test를 따로 `get_dummies` 해서 열 구성이 달라질 수 있었습니다 → test 열을 train에 맞춰 정렬(`reindex`).
XGBoost가 특수문자 열 이름을 거부할 수 있어서 열 이름을 `f0, f1, …`로 바꾸고 원래 이름은 `names`에 보관합니다.

In [ ]:
def make_data(bfnum='drop', dist='top5'):
    tr, te = train_raw.copy(), test_raw.copy()
    y = tr.pop('box_off_num')                    # 정답(관객수) 떼어 두기

    # ── 공통 ──
    for df in (tr, te):
        df.drop(['title', 'director'], axis=1, inplace=True)
        df['년'] = df['release_time'].str[:4].astype(int)
        df['월'] = df['release_time'].str[5:7].astype(int)
        df.drop('release_time', axis=1, inplace=True)

    # ── 옵션 ① dir_prev_bfnum ──
    if bfnum == 'drop':
        tr.drop('dir_prev_bfnum', axis=1, inplace=True)
        te.drop('dir_prev_bfnum', axis=1, inplace=True)
    else:  # 'fill0'
        for df in (tr, te):
            df['dir_prev_bfnum'] = np.log1p(df['dir_prev_bfnum'].fillna(0))

    # ── 옵션 ② 배급사 ──
    if dist == 'top5':
        top5 = tr['distributor'].value_counts().index[:5]
        for df in (tr, te):
            df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')
    else:  # 'rank'
        rank = y.groupby(tr['distributor']).median().rank()
        for df in (tr, te):
            df['distributor'] = df['distributor'].map(rank).fillna(0)

    # ── 원핫 인코딩 + test 열을 train에 맞추기 ──
    tr = pd.get_dummies(tr, dtype=int)
    te = pd.get_dummies(te, dtype=int).reindex(columns=tr.columns, fill_value=0)
    names = list(tr.columns)
    tr.columns = te.columns = [f'f{i}' for i in range(tr.shape[1])]
    return tr, y, te, names

---
## 2. 모델 만들기 & 점수 매기기 함수

- `get_model()` : 이름(`'rf'`, `'lgbm'`, `'xgb'`, `'cat'`)을 넣으면 모델을 만들어 줍니다. 파라미터를 안 주면 **기본값**.
- `run()` : 한 설정을 돌려서 **로컬 점수**와 **test 예측값**을 돌려줍니다.
  - 타깃 `'raw'` = 관객수 그대로 맞히기 / `'log'` = `log1p`로 숫자 범위를 줄여서 맞히기 (1,000명 → 약 6.9, 1,000만 명 → 약 16.1). 예측 후 `expm1`로 원래 관객수로 되돌립니다.
  - 5-Fold 교차검증으로 점수를 매긴 뒤, **train 전체로 다시 학습**해서 test를 예측합니다. 음수 관객수는 0으로 고칩니다.
- `save()` : 예측값을 제출 양식에 넣어 csv로 저장합니다.

In [ ]:
def get_model(name, params=None):
    params = params or {}
    if name == 'rf':
        return RandomForestRegressor(n_estimators=100, random_state=SEED, **params)   # 베이스라인 모델
    if name == 'lgbm':
        return LGBMRegressor(random_state=SEED, verbose=-1, **params)
    if name == 'xgb':
        return XGBRegressor(random_state=SEED, **params)
    return CatBoostRegressor(random_seed=SEED, verbose=0, **params)

def run(model_name, target, X, y, X_te, params=None):
    y_fit = np.log1p(y) if target == 'log' else y
    inv = np.expm1 if target == 'log' else (lambda p: p)

    oof = np.zeros(len(y))                                   # 모의고사 답안
    for tr_idx, va_idx in KFold(5, shuffle=True, random_state=SEED).split(X):
        m = get_model(model_name, params)
        m.fit(X.iloc[tr_idx], y_fit.iloc[tr_idx])            # 4조각으로 공부
        oof[va_idx] = inv(m.predict(X.iloc[va_idx]))         # 1조각으로 시험
    rmse = mean_squared_error(y, oof) ** 0.5

    m = get_model(model_name, params)                        # 전체로 다시 학습 → test 예측
    m.fit(X, y_fit)
    pred = np.clip(inv(m.predict(X_te)), 0, None)
    return rmse, pred, m

def save(pred, fname):
    sub = submission.copy()
    sub['box_off_num'] = pred
    sub.to_csv(fname, index=False)

MODELS = ['lgbm', 'xgb', 'cat']
NAME = {'rf': '랜덤포레스트', 'lgbm': 'LightGBM', 'xgb': 'XGBoost', 'cat': 'CatBoost'}
records = []   # 모든 결과를 모아 둘 곳

---
# Part 1. 모델만 바꾸기 (맡은 역할)

**베이스라인 전처리를 그대로** 쓰고(`drop`, `top5`, 관객수 그대로, 기본 설정), 모델만 바꿉니다.
전처리가 똑같기 때문에 점수 차이는 **순수하게 모델 차이**입니다. 조원들도 같은 조건으로 돌렸다면 이 표끼리 바로 비교할 수 있습니다.

| 파일 | 뜻 |
|---|---|
| `rf_baseline.csv` | 베이스라인 (랜덤포레스트) – 기준선 |
| `lgbm_raw_drop_top5.csv` | 같은 전처리 + LightGBM |
| `xgb_raw_drop_top5.csv` | 같은 전처리 + XGBoost |
| `cat_raw_drop_top5.csv` | 같은 전처리 + CatBoost |

In [ ]:
X, y, X_te, _ = make_data('drop', 'top5')     # 베이스라인 전처리

for model_name in ['rf'] + MODELS:
    rmse, pred, _ = run(model_name, 'raw', X, y, X_te)
    fname = 'rf_baseline.csv' if model_name == 'rf' else f'{model_name}_raw_drop_top5.csv'
    save(pred, fname)
    records.append({'파트': 'Part1 모델만 변경', '모델': NAME[model_name], '단계': 'S0 베이스라인 전처리',
                    '파일': fname, 'CV_RMSE': round(rmse)})
    print(f'{NAME[model_name]:8s} 로컬 RMSE = {rmse:,.0f}   → {fname}')

base_cv = records[0]['CV_RMSE']
part1 = pd.DataFrame(records)
part1['베이스라인 대비'] = (base_cv - part1['CV_RMSE']) / base_cv
part1.style.format({'CV_RMSE': '{:,}', '베이스라인 대비': '{:+.1%}'})

---
# Part 2. 한 번에 하나씩 개선하기

Part 1의 상태(S0)에서 출발해, **한 단계에 하나씩만** 바꿉니다. 바뀐 게 하나뿐이라 점수가 달라지면 그 변화 때문이라고 말할 수 있습니다.

| 단계 | 바꾸는 것 | 이유 (데이터에서 나온 가설) |
|---|---|---|
| S0 | 베이스라인 전처리 (Part 1) | 출발점 |
| S1 | `dir_prev_bfnum` 삭제 → **0으로 채우기** | 빈칸 = 데뷔작이라는 의미 있는 정보. 감독 흥행 기록은 좋은 힌트일 수 있음 |
| S2 | 배급사 top5 → **관객수 순위(rank)** | 169개 배급사 각각의 흥행력을 숫자 하나에 담으면 정보를 더 살릴 수 있음 |
| S3 | 타깃 관객수 그대로 → **log 변환** | 관객수 차이가 너무 커서 범위를 줄이면 배우기 쉬울 수 있음. 단, 대박 영화를 작게 예측할 위험도 있음 |

세 모델 모두 같은 순서로 진행합니다. 각 단계 파일은 이름만 보면 어떤 설정인지 알 수 있습니다(예: `cat_raw_fill0_rank.csv`).

In [ ]:
steps = [
    ('S1 + 데뷔작 0 채우기', 'fill0', 'top5', 'raw'),
    ('S2 + 배급사 순위',     'fill0', 'rank', 'raw'),
    ('S3 + log 타깃',        'fill0', 'rank', 'log'),
]

for model_name in MODELS:
    prev = [r for r in records if r['모델'] == NAME[model_name]][0]['CV_RMSE']   # S0 점수
    print(f'\n[{NAME[model_name]}]  S0 베이스라인 전처리        로컬 RMSE = {prev:,}')
    for step, bfnum, dist, target in steps:
        X, y, X_te, _ = make_data(bfnum, dist)
        rmse, pred, _ = run(model_name, target, X, y, X_te)
        fname = f'{model_name}_{target}_{bfnum}_{dist}.csv'
        save(pred, fname)
        records.append({'파트': 'Part2 단계별 개선', '모델': NAME[model_name], '단계': step,
                        '파일': fname, 'CV_RMSE': round(rmse)})
        diff = prev - rmse
        change = f'{abs(diff):,.0f} 좋아짐' if diff > 0 else f'{abs(diff):,.0f} 나빠짐'
        print(f'  {step:22s} 로컬 RMSE = {rmse:,.0f}  (이전 단계보다 {change})  → {fname}')
        prev = rmse

### 단계별 로컬 점수 한눈에 보기
행은 단계, 열은 모델입니다. 위에서 아래로 내려가며 숫자가 작아지면(좋아지면) 그 단계가 효과가 있었다는 뜻입니다.

⚠️ **로컬 점수만 믿으면 안 됩니다.** 데이터가 600편뿐이라 로컬 점수와 리더보드 점수가 반대로 나오는 경우가 있었습니다(예: CatBoost는 로컬에서 S1 효과가 작았지만 리더보드에서는 S1·S2를 적용한 파일이 1등). 그래서 Part 4에서 리더보드 점수와 함께 봅니다.

In [ ]:
steps_df = pd.DataFrame([r for r in records if r['모델'] != '랜덤포레스트'])
pivot = steps_df.pivot(index='단계', columns='모델', values='CV_RMSE')[[NAME[m] for m in MODELS]]
pivot.style.format('{:,}').highlight_min(color='lightgreen')   # 모델별 최저(최고 성능) 칸에 색칠

---
# Part 3. 하이퍼파라미터 튜닝

**하이퍼파라미터**는 모델의 세부 설정입니다(라면으로 치면 불 세기 · 끓이는 시간). 지금까지는 모두 기본값이었습니다.

모델마다 S2(관객수 그대로)와 S3(log) 중 **로컬 점수가 더 좋은 쪽**을 골라, 아래 후보 설정을 모두 시험합니다(모델당 16개).

| 설정 | 뜻 |
|---|---|
| `n_estimators` / `iterations` | 나무를 몇 개 쌓을지. 너무 많으면 train을 외워 버림(과적합) |
| `learning_rate` | 한 번에 얼마나 크게 고칠지 |
| `num_leaves` / `max_depth` / `depth` | 나무 하나의 복잡도. 데이터가 적으면 너무 복잡하지 않게 |
| `min_child_samples` | 나무 가지 끝에 최소 몇 편이 있어야 하는지 |
| `subsample` | 나무 하나를 만들 때 데이터를 몇 %만 쓸지 |
| `l2_leaf_reg` | CatBoost의 과적합 방지 강도 |

※ 튜닝 후 점수가 오히려 나빠질 수도 있습니다. 기본값이 이미 잘 맞았다는 뜻입니다.

In [ ]:
grids = {
    'lgbm': {'n_estimators': [300, 1000], 'learning_rate': [0.01, 0.05],
             'num_leaves': [7, 31], 'min_child_samples': [5, 20]},
    'xgb':  {'n_estimators': [300, 1000], 'learning_rate': [0.01, 0.05],
             'max_depth': [3, 6], 'subsample': [0.8, 1.0]},
    'cat':  {'iterations': [300, 1000], 'learning_rate': [0.03, 0.1],
             'depth': [4, 6], 'l2_leaf_reg': [1, 3]},
}

X, y, X_te, _ = make_data('fill0', 'rank')
for model_name in MODELS:
    s2 = [r for r in records if r['모델'] == NAME[model_name] and r['단계'].startswith('S2')][0]
    s3 = [r for r in records if r['모델'] == NAME[model_name] and r['단계'].startswith('S3')][0]
    target = 'raw' if s2['CV_RMSE'] <= s3['CV_RMSE'] else 'log'
    before = min(s2['CV_RMSE'], s3['CV_RMSE'])

    best_rmse, best_params, best_pred = np.inf, None, None
    for params in ParameterGrid(grids[model_name]):
        rmse, pred, _ = run(model_name, target, X, y, X_te, params)
        if rmse < best_rmse:
            best_rmse, best_params, best_pred = rmse, params, pred

    fname = f'{model_name}_{target}_fill0_rank_tuned.csv'
    save(best_pred, fname)
    records.append({'파트': 'Part3 튜닝', '모델': NAME[model_name], '단계': f'S4 튜닝 ({target})',
                    '파일': fname, 'CV_RMSE': round(best_rmse), '파라미터': str(best_params)})
    print(f'{NAME[model_name]:8s} ({target}) 튜닝 전 {before:,} → 튜닝 후 {best_rmse:,.0f}  {best_params}')

---
# Part 4. 결과 정리 (로컬 점수 + 리더보드 점수)

`KNOWN_LB`에는 지금까지 데이콘에 제출해서 확인한 점수를 적어 두었습니다. **새로 제출한 파일이 있으면 같은 형식으로 한 줄 추가**하고 이 셀을 다시 실행하면 표가 갱신됩니다.

결과는 `부스팅_단계별_결과.csv`로 저장됩니다(엑셀에서 한글이 깨지지 않게 `utf-8-sig`).

In [ ]:
KNOWN_LB = {   # 데이콘 리더보드 점수 (제출 후 추가하세요)
    'rf_baseline.csv':               1417427.81,
    'cat_raw_fill0_rank.csv':        1277639.76,
    'cat_log_fill0_rank.csv':        1306681.18,
    'lgbm_raw_fill0_rank.csv':       1320539.80,
    'xgb_log_fill0_rank.csv':        1366652.74,
    'xgb_log_fill0_rank_tuned.csv':  1286295.23,
    # 'lgbm_raw_drop_top5.csv':      ,
    # 'xgb_raw_drop_top5.csv':       ,
    # 'cat_raw_drop_top5.csv':       ,
}

result = pd.DataFrame(records)
result['파라미터'] = result.get('파라미터', pd.Series(dtype=str)).fillna('기본값')
result['LB_점수'] = result['파일'].map(KNOWN_LB).round()
base_lb = KNOWN_LB['rf_baseline.csv']
result['LB 베이스라인 대비'] = (base_lb - result['LB_점수']) / base_lb
result.to_csv('부스팅_단계별_결과.csv', index=False, encoding='utf-8-sig')

result[['파트', '모델', '단계', '파일', 'CV_RMSE', 'LB_점수', 'LB 베이스라인 대비']].style.format(
    {'CV_RMSE': '{:,}', 'LB_점수': '{:,.0f}', 'LB 베이스라인 대비': '{:+.1%}'}, na_rep='미제출')

### 지금까지의 결론 (리더보드 기준)
| 순위 | 파일 | 단계 | 리더보드 RMSE | 베이스라인 대비 |
|---|---|---|---|---|
| 1 | `cat_raw_fill0_rank` | CatBoost S2 | **1,277,640** | **+9.9%** |
| 2 | `xgb_log_fill0_rank_tuned` | XGBoost S4 | 1,286,295 | +9.3% |
| 3 | `cat_log_fill0_rank` | CatBoost S3 | 1,306,681 | +7.8% |
| 4 | `lgbm_raw_fill0_rank` | LightGBM S2 | 1,320,540 | +6.8% |
| 5 | `xgb_log_fill0_rank` | XGBoost S3 | 1,366,653 | +3.6% |
| – | `rf_baseline` | 기준선 | 1,417,428 | – |

1. **S1(데뷔작 0 채우기) + S2(배급사 순위)가 가장 큰 효과.** 이 두 단계를 적용한 파일이 모델과 상관없이 모두 베이스라인을 이겼습니다.
2. **log 타깃(S3)은 모델마다 다름.** CatBoost · LightGBM은 관객수 그대로가, XGBoost는 log가 더 좋았습니다. log는 대박 영화를 작게 예측하는 경향이 있습니다.
3. **튜닝은 XGBoost에서 효과가 컸고(약 6%), CatBoost는 튜닝 없이도 1등.**
4. **로컬 점수와 리더보드 점수는 다를 수 있음.** 데이터가 600편뿐이라 리더보드로 꼭 확인해야 합니다.

**최종 제출 파일:** `cat_raw_fill0_rank.csv` (Part 1 파일 3개의 리더보드 점수가 나오면 '모델만 바꿨을 때' 비교도 완성됩니다.)